In [ ]:
import re
from datetime import datetime
from collections import defaultdict
import pandas as pd
import matplotlib.pyplot as plt
import pandas as pd
import re
from datetime import datetime
from collections import defaultdict
import pandas as pd
import matplotlib.pyplot as plt

Input file is access.log which is automatically created by Apache routing configuration

### Step 1#  

Organize the requests as such:
- - a record of every request in a chronological order, 
- - one column is ip address, 
- - second column is date, 
- - third column is time, 
- - then add type ( GET, POST..), 
- - then add the other details.

In [ ]:
import re
import pandas as pd

# Path to of the input file: access.log
logfile = "access.log"

# Regex for Apache Combined Log Format (CLF)
pattern = re.compile(
    r'(?P<ip>\S+) - - \[(?P<date>\d{2}/[A-Za-z]{3}/\d{4}):(?P<time>\d{2}:\d{2}:\d{2}) [^\]]+\] '
    r'"(?P<method>\S+) (?P<request>\S+) (?P<protocol>[^"]+)" '
    r'(?P<status>\d{3}) (?P<bytes>\S+)'
)

records = []

with open(logfile, "r") as f:
    for line in f:
        m = pattern.search(line)
        if m:
            rec = m.groupdict()

            # Filter by date ≥ 01/Jun/2025
            dt = pd.to_datetime(rec["date"], format="%d/%b/%Y")
            if dt >= pd.Timestamp("2025-06-01"):
                records.append(rec)

# Build DataFrame
df = pd.DataFrame(records)
##removing local requests from the server itself. 
df = df[~df["ip"].str.startswith("127.")]


# Ensure chronological order
df["datetime"] = pd.to_datetime(df["date"] + " " + df["time"], format="%d/%b/%Y %H:%M:%S")
df = df.sort_values("datetime").reset_index(drop=True)

df


In [ ]:
df["method"].value_counts()

In [ ]:
df.head()

In [ ]:
df_original = df.copy()

In [ ]:
valid_ip = df[df["request"].str.contains("logo", case=False, na=False)]["ip"].unique()

# Ignoring Bots at first

1) total unique users per day ( x-axis is time)

2) for the unique users at all time, plot a scatter plot of the first request by each unique user. 

3) total duration of reqeusst per day, 

4) average request duration per day, 

5) then also plot a historgram of average request duration by user, and 

6) histrogram of total nb of user per day, 7) a cumulative plot of unique users increase (unique over all duration) per day, 

8) pie chart of type of requests (GET vs post) this means they only viewed, versus they tested it. 

In [ ]:

#1) Total unique users per day
df["day"] = df["datetime"].dt.date
users_per_day = df.groupby("day")["ip"].nunique()

users_per_day.plot(figsize=(10,4))
plt.title("Unique Users per Day")
plt.ylabel("Users")
plt.xlabel("Day")
plt.show()


#2) Scatter plot of first request per unique user
df = df.sort_values("datetime")

# Now compute earliest request per user (already in order)
first_req = df.groupby("ip", sort=False)["datetime"].min().reset_index()

# Then sort by that min datetime
first_req = first_req.sort_values("datetime").reset_index(drop=True)

plt.figure(figsize=(10,4))
plt.scatter(first_req["datetime"], range(len(first_req)), s=5)
plt.title("First Request Time per User")
plt.xlabel("Datetime")
plt.ylabel("User Index")
plt.show()


df = df.sort_values(["ip", "datetime"])
df["duration"] = df.groupby("ip")["datetime"].diff().dt.total_seconds()

#3) Total duration of requests per day
total_duration_per_day = df.groupby("day")["duration"].sum()

total_duration_per_day.plot(figsize=(10,4))
plt.title("Total Duration Per Day")
plt.ylabel("Seconds")
plt.xlabel("Day")
plt.show()

#✅ 4) Average request duration per day
avg_duration_per_day = df.groupby("day")["duration"].mean()

avg_duration_per_day.plot(figsize=(10,4))
plt.title("Average Request Duration Per Day")
plt.ylabel("Seconds")
plt.xlabel("Day")
plt.show()

#✅ 5) Histogram — average request duration per user
avg_duration_per_user = df.groupby("ip")["duration"].mean()

avg_duration_per_user.hist(bins=30)
plt.title("Histogram: Average Duration per User")
plt.xlabel("Seconds")
plt.ylabel("Count")
plt.show()

#✅ 6) Histogram — total number of users per day
users_per_day.hist(bins=15)
plt.title("Histogram: Users per Day")
plt.xlabel("Users")
plt.ylabel("Count of Days")
plt.show()

#✅ 7) Cumulative unique users over time
unique_users = (
    df.sort_values("datetime")
      .drop_duplicates("ip")
      .groupby(df["datetime"].dt.date)["ip"]
      .count()
      .cumsum()
)

unique_users.plot(figsize=(10,4))
plt.title("Cumulative Unique Users Over Time")
plt.ylabel("Total Users")
plt.xlabel("Day")
plt.show()

#✅ 8) Pie chart of GET vs POST
'''
df["method"].value_counts().plot.pie(
    autopct="%1.1f%%",
    figsize=(6,6)
)
plt.title("GET vs POST Requests")
plt.ylabel("")
plt.show()
'''
df = df[df["method"].isin(["GET", "POST"])]

label_map = {
    "GET": "Viewing information",
    "POST": "Interactive browsing"
}

method_counts = df["method"].map(label_map).value_counts()

method_counts.plot.pie(
    autopct="%1.1f%%",
    figsize=(5,5)
)

plt.title("Request Types: Viewing vs Interactive")
plt.ylabel("")
plt.show()

## Visualizations after Cleaning Bots

Tested methods of common bot-cleaning options (beyond filtering by GET/POST):

- User-Agent filtering → drop rows where user_agent contains known bots (curl, python-requests, Googlebot, etc.).

- Request-frequency rule → remove IPs making too many requests per second.

- Path pattern filtering → remove hits to /wp-login.php, /xmlrpc.php, /phpMyAdmin/, etc.

- Invalid status spikes → remove IPs producing mostly 404/403/500.

- No-cookie behavior → repeated GETs with no sessions often = bots.

- Random query params → junk URLs, long random strings.

- Headless signatures → “HeadlessChrome”, “ScreamingFrog”, etc.

In [ ]:
#Removing all but GET or POST
df = df[df["method"].isin(["GET", "POST"])]
df.columns

In [ ]:
# 1) Keep only GET + POST
df = df[df["method"].isin(["GET", "POST"])]

# 2) Remove high-frequency IPs (more than 1 request/sec repeatedly)
df = df.sort_values(["ip", "datetime"])
req_rate = df.groupby("ip")["datetime"].diff().dt.total_seconds()
bad_ips = df["ip"][req_rate < 1].unique()
df = df[~df["ip"].isin(bad_ips)]

# 3) Remove paths typical for bots
bad_paths = [
    "/wp-login", "/wp-admin", "/xmlrpc.php",
    "/admin", "/phpmyadmin", "/phpMyAdmin",
    "/cgi-bin/", "/.env", "/server-status"
]
pattern = "|".join(bad_paths)
df = df[~df["request"].str.contains(pattern, case=False, na=False)]

# 4) Remove IPs with mostly error statuses
error_codes = [400, 401, 403, 404, 429, 500, 501, 502, 503]
ip_error_ratio = (
    df.assign(error=df["status"].isin(error_codes))
      .groupby("ip")["error"]
      .mean()
)
bad_ips = ip_error_ratio[ip_error_ratio > 0.8].index
df = df[~df["ip"].isin(bad_ips)]

# 5) Remove requests with extremely long URLs
df = df[df["request"].str.len() < 200]

# df is now cleaned
df_clean = df.copy()
# Duration in minutes
df_clean["duration_min"] = df_clean["duration"] / 60

# Duration in hours
df_clean["duration_hr"] = df_clean["duration"] / 3600


In [ ]:
df_long = df_clean[df_clean["duration"]>60]

In [ ]:
import matplotlib.pyplot as plt

# 1) Total unique users per day
df_long["day"] = df_long["datetime"].dt.date
users_per_day = df_long.groupby("day")["ip"].nunique()

users_per_day.plot(figsize=(10,4), color="green")
plt.title("Unique Users per Day")
plt.ylabel("Users")
plt.xlabel("Day")
plt.show()

# 2) Scatter plot of first request per unique user
df_long = df_long.sort_values("datetime")
first_req = df_long.groupby("ip", sort=False)["datetime"].min().reset_index()
first_req = first_req.sort_values("datetime").reset_index(drop=True)

plt.figure(figsize=(10,4))
plt.scatter(first_req["datetime"], range(len(first_req)), s=5, color="green")
plt.title("First Request Time per User")
plt.xlabel("Datetime")
plt.ylabel("User Index")
plt.show()

# 3) Total duration of requests per day
df_long = df_long.sort_values(["ip", "datetime"])
df_long["duration"] = df_long.groupby("ip")["datetime"].diff().dt.total_seconds()
total_duration_per_day = df_long.groupby("day")["duration"].sum()

total_duration_per_day.plot(figsize=(10,4), color="green")
plt.title("Total Duration Per Day")
plt.ylabel("Seconds")
plt.xlabel("Day")
plt.show()

# 4) Average request duration per day
avg_duration_per_day = df_long.groupby("day")["duration"].mean()
avg_duration_per_day.plot(figsize=(10,4), color="green")
plt.title("Average Request Duration Per Day")
plt.ylabel("Seconds")
plt.xlabel("Day")
plt.show()

# 5) Histogram — average request duration per user
avg_duration_per_user = df_long.groupby("ip")["duration_min"].mean()
plt.figure(figsize=(8,4))
plt.hist(avg_duration_per_user, bins=10, color="green")
plt.title("Histogram: Average Duration per User")
plt.xlabel("Minutes")
plt.ylabel("Count")
plt.show()

# 6) Histogram — total number of users per day
plt.figure(figsize=(8,4))
plt.hist(users_per_day, bins=15, color="green")
plt.title("Histogram: Users per Day")
plt.xlabel("Users")
plt.ylabel("Count of Days")
plt.show()

# 7) Cumulative unique users over time
unique_users = (
    df_long.sort_values("datetime")
           .drop_duplicates("ip")
           .groupby(df_long["datetime"].dt.date)["ip"]
           .count()
           .cumsum()
)
unique_users.plot(figsize=(10,4), color="green")
plt.title("Cumulative Unique Users Over Time")
plt.ylabel("Total Users")
plt.xlabel("Day")
plt.show()

# 8) Pie chart of GET vs POST with custom labels
df_long_clean = df_long[df_long["method"].isin(["GET", "POST"])]
label_map = {"GET": "Viewing information", "POST": "Interactive browsing"}
method_counts = df_long_clean["method"].map(label_map).value_counts()

method_counts.plot.pie(
    autopct="%1.1f%%",
    figsize=(5,5),
    colors=["green", "lightgreen"]
)
plt.title("Request Types: Viewing vs Interactive")
plt.ylabel("")
plt.show()


# Looking at valid human IPs only

In [ ]:
df_clean.shape

In [ ]:
df_original.shape

In [ ]:
#df_long.shape

In [ ]:
#df = df_original[df_original["ip"].isin(valid_ip)]
df = df_clean[df_clean["ip"].isin(valid_ip)]

In [ ]:
df.shape

## DEFINE sessions

In [ ]:
df = df[df["method"].isin(["GET", "POST"])]


In [ ]:

# Sort df_clean by ip and datetime
df = df.sort_values(["ip", "datetime"]).reset_index(drop=True)

sessions = []
session_id = 0

# Loop through each IP
for ip, df_ip in df.groupby("ip"):
    df_ip = df_ip.reset_index(drop=True)
    i = 0
    n = len(df_ip)

    while i < n:
        row = df_ip.loc[i]
        # Start new session
        session_id += 1
        session_start = row["datetime"]
        session_end = row["datetime"]
        post_flag = 1 if row["method"] == "POST" else 0
        requests_in_session = 1  # optional, can count number of requests

        j = i + 1
        while j < n:
            next_row = df_ip.loc[j]
            time_diff = (next_row["datetime"] - session_end).total_seconds()

            # Start a new session if first-request pattern OR gap >1hr
            if next_row["request"].startswith("GET / HTTP/1.1") or time_diff > 3600:
                break

            # Otherwise, continue current session
            session_end = next_row["datetime"]
            if next_row["method"] == "POST":
                post_flag = 1
            requests_in_session += 1
            j += 1

        # Save session
        sessions.append({
            "session_id": session_id,
            "ip": ip,
            "session_start": session_start,
            "session_end": session_end,
            "duration_sec": (session_end - session_start).total_seconds(),
            "post_flag": post_flag,
            "num_requests": requests_in_session
        })

        # Move to next session
        i = j

# Build DataFrame
df_sessions = pd.DataFrame(sessions)
df_sessions.head()


In [ ]:
df_sessions.shape

In [ ]:
len(df_sessions["ip"].unique())


## Visualizations

In [ ]:
#1. Session count and activity over time

#a) Total sessions per day (bar plot)

df_sessions['day'] = df_sessions['session_start'].dt.date
sessions_per_day = df_sessions.groupby('day')['session_id'].count()
sessions_per_day.plot(kind='bar', figsize=(12,5), color='skyblue')
plt.title("Total Sessions Per Day")
plt.ylabel("Number of Sessions")
plt.xlabel("Day")
plt.xticks(rotation=45)
plt.show()


#b) Total unique users per day (bar plot)

unique_users_per_day = df_sessions.groupby('day')['ip'].nunique()
unique_users_per_day.plot(kind='bar', figsize=(12,5), color='lightgreen')
plt.title("Total Unique Users Per Day")
plt.ylabel("Number of Users")
plt.xlabel("Day")
plt.xticks(rotation=45)
plt.show()


#c) Histogram of sessions per day (frequency distribution)

plt.figure(figsize=(10,4))
plt.hist(df_sessions['day'], bins=len(df_sessions['day'].unique()), color='orange')
plt.title("Histogram: Sessions Per Day")
plt.xlabel("Day")
plt.ylabel("Number of Sessions")
plt.xticks(rotation=45)
plt.show()

#2. Session duration analysis

#a) Histogram of session durations (minutes)

df_sessions['duration_min'] = df_sessions['duration_sec'] / 60
plt.figure(figsize=(10,4))
plt.hist(df_sessions['duration_min'], bins=30, color='purple', alpha=0.7)
plt.title("Histogram: Session Durations (minutes)")
plt.xlabel("Duration (minutes)")
plt.ylabel("Number of Sessions")
plt.show()


#b) Boxplot of session durations by day

import seaborn as sns
plt.figure(figsize=(12,5))
sns.boxplot(x='day', y='duration_min', data=df_sessions, palette='Set2')
plt.title("Session Durations by Day")
plt.xlabel("Day")
plt.ylabel("Duration (minutes)")
plt.xticks(rotation=45)
plt.show()


#c) Cumulative distribution of session durations

df_sessions['duration_min'].sort_values().plot(kind='line', figsize=(10,4), color='teal')
plt.title("Cumulative Distribution of Session Durations")
plt.xlabel("Sessions (sorted)")
plt.ylabel("Duration (minutes)")
plt.show()

#3. POST activity

#a) Pie chart: POST vs GET-only

post_counts = df_sessions['post_flag'].value_counts()
post_counts.index = ['GET-only', 'POST']
post_counts.plot.pie(autopct="%1.1f%%", colors=['skyblue','orange'], figsize=(6,6))
plt.title("Sessions: GET-only vs POST")
plt.ylabel("")
plt.show()


#b) Average number of requests per session, colored by POST flag

avg_requests_post = df_sessions.groupby('post_flag')['num_requests'].mean()
avg_requests_post.plot(kind='bar', color=['skyblue','orange'], figsize=(6,4))
plt.title("Average Number of Requests per Session")
plt.xticks([0,1], ['GET-only','POST'], rotation=0)
plt.ylabel("Average Requests")
plt.show()

#4. Requests per session

#a) Histogram of number of requests per session

plt.figure(figsize=(10,4))
plt.hist(df_sessions['num_requests'], bins=30, color='green', alpha=0.7)
plt.title("Histogram: Number of Requests per Session")
plt.xlabel("Requests")
plt.ylabel("Number of Sessions")
plt.show()


#b) Scatter plot: session duration vs number of requests

plt.figure(figsize=(10,4))
plt.scatter(df_sessions['num_requests'], df_sessions['duration_min'], c='purple', alpha=0.6)
plt.title("Session Duration vs Number of Requests")
plt.xlabel("Number of Requests")
plt.ylabel("Duration (minutes)")
plt.show()


#c) Boxplot of requests per session by day

plt.figure(figsize=(12,5))
sns.boxplot(x='day', y='num_requests', data=df_sessions, palette='Set3')
plt.title("Number of Requests per Session by Day")
plt.xlabel("Day")
plt.ylabel("Number of Requests")
plt.xticks(rotation=45)
plt.show()

#5. User-level summary

#a) Total sessions per user

total_sessions_user = df_sessions.groupby('ip')['session_id'].count()
plt.figure(figsize=(10,4))
plt.hist(total_sessions_user, bins=30, color='skyblue', alpha=0.7)
plt.title("Total Sessions per User")
plt.xlabel("Number of Sessions")
plt.ylabel("Number of Users")
plt.show()


#b) Average session duration per user

avg_duration_user = df_sessions.groupby('ip')['duration_min'].mean()
plt.figure(figsize=(10,4))
plt.hist(avg_duration_user, bins=30, color='orange', alpha=0.7)
plt.title("Average Session Duration per User")
plt.xlabel("Duration (minutes)")
plt.ylabel("Number of Users")
plt.show()


#c) Total requests per user

total_requests_user = df_sessions.groupby('ip')['num_requests'].sum()
plt.figure(figsize=(10,4))
plt.scatter(total_requests_user.index, total_requests_user.values, color='green', alpha=0.6)
plt.title("Total Requests per User")
plt.xlabel("User Index")
plt.ylabel("Total Requests")
plt.show()

#6. Temporal patterns

#a) Sessions heatmap: day vs hour of day

df_sessions['hour'] = df_sessions['session_start'].dt.hour
heatmap_data = df_sessions.groupby(['day','hour'])['session_id'].count().unstack(fill_value=0)
plt.figure(figsize=(12,6))
sns.heatmap(heatmap_data, cmap='YlGnBu')
plt.title("Sessions Heatmap: Day vs Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Day")
plt.show()


#b) Stacked bar chart: GET-only vs POST sessions per day

stacked_data = df_sessions.groupby(['day','post_flag'])['session_id'].count().unstack(fill_value=0)
stacked_data.columns = ['GET-only','POST']
stacked_data.plot(kind='bar', stacked=True, figsize=(12,5), color=['skyblue','orange'])
plt.title("GET-only vs POST Sessions per Day")
plt.ylabel("Number of Sessions")
plt.xlabel("Day")
plt.xticks(rotation=45)
plt.show()


#c) Cumulative unique users over time

unique_users_cum = df_sessions.sort_values('session_start').drop_duplicates('ip').groupby(df_sessions['session_start'].dt.date)['ip'].count().cumsum()
unique_users_cum.plot(figsize=(12,4), color='purple')
plt.title("Cumulative Unique Users Over Time")
plt.xlabel("Day")
plt.ylabel("Total Users")
plt.show()

#7. Session lifecycle / sequences (Gantt-style)
plt.figure(figsize=(12,6))

for i, (ip, group) in enumerate(df_sessions.groupby('ip')):
    for _, row in group.iterrows():
        plt.hlines(y=i, xmin=row['session_start'], xmax=row['session_end'],
                   color='skyblue', linewidth=4)

plt.title("Session Timeline per User")
plt.xlabel("Datetime")
plt.ylabel("User Index")
plt.show()

#8. POST fraction vs session length
plt.figure(figsize=(10,5))
sns.boxplot(x='post_flag', y='duration_min', data=df_sessions, palette=['skyblue','orange'])
plt.xticks([0,1], ['GET-only','POST'])
plt.title("Session Duration by POST Flag")
plt.ylabel("Duration (minutes)")
plt.xlabel("Session Type")
plt.show()

In [ ]:
print(df.shape)

In [ ]:

#1) Total unique users per day
df["day"] = df["datetime"].dt.date
users_per_day = df.groupby("day")["ip"].nunique()

users_per_day.plot(figsize=(10,4))
plt.title("Unique Users per Day")
plt.ylabel("Users")
plt.xlabel("Day")
plt.show()


#2) Scatter plot of first request per unique user
df = df.sort_values("datetime")

# Now compute earliest request per user (already in order)
first_req = df.groupby("ip", sort=False)["datetime"].min().reset_index()

# Then sort by that min datetime
first_req = first_req.sort_values("datetime").reset_index(drop=True)

plt.figure(figsize=(10,4))
plt.scatter(first_req["datetime"], range(len(first_req)), s=5)
plt.title("First Request Time per User")
plt.xlabel("Datetime")
plt.ylabel("User Index")
plt.show()


df = df.sort_values(["ip", "datetime"])
df["duration"] = df.groupby("ip")["datetime"].diff().dt.total_seconds()

#3) Total duration of requests per day
total_duration_per_day = df.groupby("day")["duration"].sum()

total_duration_per_day.plot(figsize=(10,4))
plt.title("Total Duration Per Day")
plt.ylabel("Seconds")
plt.xlabel("Day")
plt.show()

#✅ 4) Average request duration per day
avg_duration_per_day = df.groupby("day")["duration"].mean()

avg_duration_per_day.plot(figsize=(10,4))
plt.title("Average Request Duration Per Day")
plt.ylabel("Seconds")
plt.xlabel("Day")
plt.show()

#✅ 5) Histogram — average request duration per user
avg_duration_per_user = df.groupby("ip")["duration"].mean()

avg_duration_per_user.hist(bins=30)
plt.title("Histogram: Average Duration per User")
plt.xlabel("Seconds")
plt.ylabel("Count")
plt.show()

#✅ 6) Histogram — total number of users per day
users_per_day.hist(bins=15)
plt.title("Histogram: Users per Day")
plt.xlabel("Users")
plt.ylabel("Count of Days")
plt.show()

#✅ 7) Cumulative unique users over time
unique_users = (
    df.sort_values("datetime")
      .drop_duplicates("ip")
      .groupby(df["datetime"].dt.date)["ip"]
      .count()
      .cumsum()
)

unique_users.plot(figsize=(10,4))
plt.title("Cumulative Unique Users Over Time")
plt.ylabel("Total Users")
plt.xlabel("Day")
plt.show()

#✅ 8) Pie chart of GET vs POST
'''
df["method"].value_counts().plot.pie(
    autopct="%1.1f%%",
    figsize=(6,6)
)
plt.title("GET vs POST Requests")
plt.ylabel("")
plt.show()
'''
df = df[df["method"].isin(["GET", "POST"])]

label_map = {
    "GET": "Viewing information",
    "POST": "Interactive browsing"
}

method_counts = df["method"].map(label_map).value_counts()

method_counts.plot.pie(
    autopct="%1.1f%%",
    figsize=(5,5)
)

plt.title("Request Types: Viewing vs Interactive")
plt.ylabel("")
plt.show()

In [ ]:
df_sessions.shape

In [ ]:
len(df_sessions["ip"].unique())

#### Selected visualizations

In [ ]:
#a) Total sessions per day (histogram)
df_sessions['day'] = df_sessions['session_start'].dt.date
sessions_per_day = df_sessions.groupby('day')['session_id'].count()

plt.figure(figsize=(10,5))
plt.hist(
    sessions_per_day,
    bins=10,
    color='skyblue',
    edgecolor='black',  # outline
    alpha=0.9
)
plt.title("Histogram: Total Sessions Per Day", fontsize=14)
plt.xlabel("Number of Sessions", fontsize=12)
plt.ylabel("Number of Days", fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()



#b) Total unique users per day (histogram)
unique_users_per_day = df_sessions.groupby('day')['ip'].nunique()

plt.figure(figsize=(10,5))
plt.hist(
    unique_users_per_day,
    bins=10,
    color='lightgreen',
    edgecolor='black',  # outline
    alpha=0.9
)
plt.title("Histogram: Total Unique Users Per Day", fontsize=14)
plt.xlabel("Number of Users", fontsize=12)
plt.ylabel("Number of Days", fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


In [ ]:
#a) Pie chart: POST vs GET-only

# Map post_flag to descriptive labels
import matplotlib.pyplot as plt

# Map post_flag to descriptive labels
label_map = {0: "Viewing Only", 1: "Interactive"}
post_counts = df_sessions['post_flag'].map(label_map).value_counts()

# Define explode to slightly separate the slices
explode = (0.05, 0.05)  # small separation for both slices

# Plot pie chart
post_counts.plot.pie(
    autopct="%1.1f%%",
    colors=['skyblue', 'orange'],
    figsize=(6,6),
    explode=explode,
    shadow=True,
    startangle=90,
    wedgeprops={'edgecolor':'black'}
)
plt.title("Sessions: Viewing Only vs Interactive", fontsize=14)
plt.ylabel("")  # remove default ylabel
plt.tight_layout()
plt.show()


In [ ]:
#b) Average session duration per user

avg_duration_user = df_sessions.groupby('ip')['duration_min'].mean()

plt.figure(figsize=(10,4))
plt.hist(
    avg_duration_user,
    bins=10,
    color='orange',
    edgecolor='black',  # outline
    alpha=0.9
)
plt.title("Average Session Duration per User", fontsize=14)
plt.xlabel("Duration (minutes)", fontsize=12)
plt.ylabel("Number of Users", fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


#a) Total sessions per user

total_sessions_user = df_sessions.groupby('ip')['session_id'].count()

plt.figure(figsize=(10,4))
plt.hist(
    total_sessions_user,
    bins=10,
    color='skyblue',
    edgecolor='black',   # outline around bars
    alpha=0.9
)
plt.title("Total Sessions per User", fontsize=14)
plt.xlabel("Number of Sessions", fontsize=12)
plt.ylabel("Number of Users", fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)  # horizontal grid lines
plt.tight_layout()
plt.show()



In [ ]:
plt.figure(figsize=(10, 3))
plt.boxplot(avg_duration_user, vert=False)
plt.xlabel("Average Session Duration (minutes)")
plt.title("Average Session Duration per User")
plt.grid(axis='x', linestyle='--', alpha=0.4)
plt.show()

In [ ]:
# Cumulative unique users over time
unique_users = (
    df.sort_values("datetime")
      .drop_duplicates("ip")
      .groupby(df["datetime"].dt.date)["ip"]
      .count()
      .cumsum()
)

plt.figure(figsize=(12,5))
plt.plot(
    unique_users.index,
    unique_users.values,
    color='purple',   # vibrant color
    linewidth=2.5,
    marker='o',
    markersize=5
)
plt.title("Cumulative Unique Users Over Time", fontsize=14, fontweight='bold')
plt.xlabel("Day", fontsize=12)
plt.ylabel("Total Users", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# 6) Histogram — total number of users per day
plt.figure(figsize=(8,4))
plt.hist(users_per_day, bins=15, color="green")
plt.title("Distribution of Number of Users per Day")
plt.xlabel("Number of Users")
plt.ylabel("Count of Days")
plt.show()

In [ ]:
plt.figure(figsize=(8,4))
plt.hist(users_per_day, bins=15, color="green")
plt.title("Distribution of Number of Users per Day")
plt.xlabel("Number of Users")
plt.ylabel("Count of Days")
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.show()